# Stateful window TVT model

Bu deney kuyuları MD sırasında işler. Sentetik veya gerçek PS öncesindeki TVT değerleri context olarak verilir; PS sonrasında TVT sıfırlanır. Geometri ve GR feature'ları bütün kuyu boyunca kullanılmaya devam eder. GRU hidden state'i bir window'dan sonrakine aktarılır ve yalnızca tahmin bölgesinde loss hesaplanır.

Sentetik anchor örneklemesi gerçek PS dağılımına hizalanmıştır: anchor'ların %70'i kuyu uzunluğunun %20–35 bandından (train setindeki gerçek PS'lerin yoğunlaştığı bölge), %30'u uniform aralıktan gelir. Minimum forecast uzunluğu kuyu uzunluğunun %40'ına oranlanır; böylece her episode uzun ufku da eğitir. Epoch başına kuyu başı `ANCHORS_PER_WELL` sentetik episode örneklenir.

In [9]:
import copy
import random
from typing import Literal

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

from training import (
    RNN_DROP_COLUMNS,
    RNN_FEATURE_COLUMNS,
    StatefulTVTWindowDataset,
    prepare_rnn_features,
)
from utils import (
    get_all_test_ids,
    get_all_training_ids,
    test_folder,
    train_folder,
)

SEED = 42
VALID_SIZE = 0.15
WINDOW_SIZE = 256
CONTEXT_LENGTH = 256
MIN_FORECAST_LENGTH = 256
MIN_FORECAST_FRACTION = 0.40
REAL_PS_ANCHOR_PROB = 0.70
REAL_PS_FRACTION_RANGE = (0.20, 0.35)
ANCHORS_PER_WELL = 4
NUM_EPOCHS = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: mps


/Users/doruksarigun/Kaggle/ROGII - Wellbore Geology Prediction/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
all_training_ids = get_all_training_ids()
reserved_test_ids = set(get_all_test_ids())
modeling_ids = sorted(set(all_training_ids) - reserved_test_ids)

train_well_ids, valid_well_ids = train_test_split(
    modeling_ids,
    test_size=VALID_SIZE,
    random_state=SEED,
    shuffle=True,
)

assert set(train_well_ids).isdisjoint(valid_well_ids)
assert set(train_well_ids).isdisjoint(reserved_test_ids)
print(f"Train wells: {len(train_well_ids)}")
print(f"Validation wells: {len(valid_well_ids)}")
print(f"Reserved test wells: {len(reserved_test_ids)}")

Train wells: 654
Validation wells: 116
Reserved test wells: 3


In [11]:
train_wells = prepare_rnn_features(
    training_ids=train_well_ids,
    feature_columns=RNN_FEATURE_COLUMNS,
    drop_columns=RNN_DROP_COLUMNS,
    imputed=True,
)
valid_wells = prepare_rnn_features(
    training_ids=valid_well_ids,
    feature_columns=RNN_FEATURE_COLUMNS,
    drop_columns=RNN_DROP_COLUMNS,
    imputed=True,
)
test_wells = prepare_rnn_features(
    training_ids=sorted(reserved_test_ids),
    feature_columns=RNN_FEATURE_COLUMNS,
    drop_columns=RNN_DROP_COLUMNS,
    imputed=True,
    data_dir=test_folder,
    target_data_dir=train_folder,
)

len(train_wells), len(valid_wells), len(test_wells)

(654, 116, 3)

In [12]:
MASK_FEATURE_COLUMNS = [
    "GR_isna",
    "GR_typewell_imputed_mask",
    "GR_neighbor_imputed_mask",
]
CONTINUOUS_FEATURE_COLUMNS = [
    column for column in RNN_FEATURE_COLUMNS
    if column not in MASK_FEATURE_COLUMNS
]

feature_scaler = StandardScaler()
target_scaler = StandardScaler()
for well in train_wells.values():
    feature_scaler.partial_fit(well.data[CONTINUOUS_FEATURE_COLUMNS])
    target_scaler.partial_fit(well.data[["TVT"]])

target_mean = float(target_scaler.mean_[0])
target_scale = float(target_scaler.scale_[0])

def apply_scalers(prepared_wells):
    for well in prepared_wells.values():
        well.data.loc[:, CONTINUOUS_FEATURE_COLUMNS] = (
            feature_scaler.transform(
                well.data[CONTINUOUS_FEATURE_COLUMNS]
            )
        )
        well.data.loc[:, "TVT_scaled"] = target_scaler.transform(
            well.data[["TVT"]]
        ).ravel()
        well.data.loc[:, "observed_TVT_scaled"] = 0.0
        observed_mask = well.data["observed_TVT"].notna()
        well.data.loc[observed_mask, "observed_TVT_scaled"] = (
            well.data.loc[observed_mask, "observed_TVT"] - target_mean
        ) / target_scale

apply_scalers(train_wells)
apply_scalers(valid_wells)
apply_scalers(test_wells)
target_mean, target_scale

(11502.671923024185, 637.698604177825)

In [13]:
def make_real_datasets(prepared_wells):
    return {
        well_id: StatefulTVTWindowDataset(
            data=well.data,
            feature_columns=RNN_FEATURE_COLUMNS,
            target_column="TVT_scaled",
            window_size=WINDOW_SIZE,
            context_length=CONTEXT_LENGTH,
            mode="real",
            prediction_mask_column="is_prediction_row",
            observed_context_column="observed_TVT_scaled",
        )
        for well_id, well in prepared_wells.items()
    }

valid_datasets = make_real_datasets(valid_wells)
test_datasets = make_real_datasets(test_wells)

example_well_id = max(
    test_datasets,
    key=lambda well_id: int(
        test_wells[well_id].data["is_prediction_row"].sum()
    ),
)
example_dataset = test_datasets[example_well_id]
print(f"Example well: {example_well_id}")
print(f"Prediction rows: {int(example_dataset.loss_mask.sum())}")
print(f"Sequential segments: {len(example_dataset)}")
print(f"First/last segment: {example_dataset.segments[0]} / {example_dataset.segments[-1]}")
print(f"Input size: {example_dataset.features.shape[-1]}")

Example well: 00bbac68
Prediction rows: 6014
Sequential segments: 30
First/last segment: (0, 256) / (7431, 7557)
Input size: 22


Model girdisi 20 temel feature, ölçeklenmiş `TVT_context` ve `TVT_known_mask` olmak üzere 22 kanaldır. `forward` hem sequence tahminini hem de sonraki window'a taşınacak hidden state'i döndürür.

In [14]:
RNN_TYPES = {"rnn": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM}

class StatefulTVTModel(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int = 64,
        rnn_type: Literal["rnn", "gru", "lstm"] = "gru",
        num_layers: int = 1,
        dropout: float = 0.2,
    ):
        super().__init__()
        if rnn_type not in RNN_TYPES:
            raise ValueError(f"Invalid RNN type: {rnn_type}")
        self.rnn = RNN_TYPES[rnn_type](
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, x, hidden=None):
        sequence_output, hidden = self.rnn(x, hidden)
        predictions = self.head(sequence_output).squeeze(-1)
        return predictions, hidden

model = StatefulTVTModel(
    input_size=len(RNN_FEATURE_COLUMNS) + 2,
    hidden_size=64,
    rnn_type="gru",
).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-3, weight_decay=1e-5
)

with torch.no_grad():
    first_segment = example_dataset[0]["features"].unsqueeze(0).to(device)
    smoke_output, smoke_hidden = model(first_segment)
assert smoke_output.shape == first_segment.shape[:2]
smoke_output.shape, smoke_hidden.shape

(torch.Size([1, 256]), torch.Size([1, 1, 64]))

In [15]:
def detach_hidden(hidden):
    if isinstance(hidden, tuple):
        return tuple(state.detach() for state in hidden)
    return hidden.detach()

def masked_mse_loss(predictions, targets, loss_mask):
    return ((predictions - targets) ** 2)[loss_mask].mean()

def sample_prediction_start(well, rng):
    """Sentetik PS'i gercek PS dagilimina agirlikli olarak orneklar.

    Anchor'larin REAL_PS_ANCHOR_PROB kadari kuyu uzunlugunun
    REAL_PS_FRACTION_RANGE bandindan (gercek PS'lerin yogunlastigi bolge),
    kalani uniform araliktan gelir. Minimum forecast uzunlugu kuyu
    uzunluguna oranlanir ki uzun ufuk her episode'da egitilsin.
    """
    num_rows = len(well.data)
    min_forecast = max(
        MIN_FORECAST_LENGTH,
        int(num_rows * MIN_FORECAST_FRACTION),
    )
    first = CONTEXT_LENGTH
    last = num_rows - min_forecast
    if last < first:
        return None

    if rng.random() < REAL_PS_ANCHOR_PROB:
        low = max(first, int(num_rows * REAL_PS_FRACTION_RANGE[0]))
        high = min(last, int(num_rows * REAL_PS_FRACTION_RANGE[1]))
        if low <= high:
            return rng.randint(low, high)

    return rng.randint(first, last)

def train_one_epoch(model, prepared_wells, optimizer, rng, epoch=None):
    model.train()
    squared_error_sum = 0.0
    observation_count = 0
    well_ids = list(prepared_wells)
    rng.shuffle(well_ids)

    progress = tqdm(
        well_ids,
        desc=f"Epoch {epoch}" if epoch is not None else "Train",
        unit="well",
        leave=False,
    )
    for well_id in progress:
        well = prepared_wells[well_id]
        for _ in range(ANCHORS_PER_WELL):
            prediction_start = sample_prediction_start(well, rng)
            if prediction_start is None:
                continue
            dataset = StatefulTVTWindowDataset(
                data=well.data,
                feature_columns=RNN_FEATURE_COLUMNS,
                target_column="TVT_scaled",
                window_size=WINDOW_SIZE,
                context_length=CONTEXT_LENGTH,
                mode="synthetic",
                prediction_start=prediction_start,
            )
            hidden = None

            for segment in dataset:
                features = segment["features"].unsqueeze(0).to(device)
                targets = segment["target"].unsqueeze(0).to(device)
                loss_mask = segment["loss_mask"].unsqueeze(0).to(device)

                if not loss_mask.any():
                    with torch.no_grad():
                        _, hidden = model(features, hidden)
                    hidden = detach_hidden(hidden)
                    continue

                optimizer.zero_grad(set_to_none=True)
                predictions, hidden = model(features, hidden)
                loss = masked_mse_loss(predictions, targets, loss_mask)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                hidden = detach_hidden(hidden)

                errors_tvt = (predictions.detach() - targets) * target_scale
                squared_error_sum += torch.sum(
                    errors_tvt[loss_mask] ** 2
                ).item()
                observation_count += int(loss_mask.sum().item())

        if observation_count > 0:
            progress.set_postfix(
                running_rmse=f"{np.sqrt(squared_error_sum / observation_count):.4f}"
            )

    if observation_count == 0:
        raise ValueError("Training epoch'unda loss satiri bulunamadi.")
    return float(np.sqrt(squared_error_sum / observation_count))

@torch.no_grad()
def evaluate_real_ps(model, datasets):
    model.eval()
    squared_error_sum = 0.0
    observation_count = 0

    for dataset in tqdm(
        datasets.values(), desc="Eval", unit="well", leave=False
    ):
        hidden = None
        for segment in dataset:
            features = segment["features"].unsqueeze(0).to(device)
            targets = segment["target"].unsqueeze(0).to(device)
            loss_mask = segment["loss_mask"].unsqueeze(0).to(device)
            predictions, hidden = model(features, hidden)
            hidden = detach_hidden(hidden)

            if loss_mask.any():
                errors_tvt = (predictions - targets) * target_scale
                squared_error_sum += torch.sum(
                    errors_tvt[loss_mask] ** 2
                ).item()
                observation_count += int(loss_mask.sum().item())

    if observation_count == 0:
        raise ValueError("Gercek PS sonrasinda skorlanacak satir yok.")
    return float(np.sqrt(squared_error_sum / observation_count))

In [16]:
history = []
best_valid_rmse = float("inf")
best_state = None
anchor_rng = random.Random(SEED)

for epoch in range(1, NUM_EPOCHS + 1):
    train_rmse = train_one_epoch(
        model, train_wells, optimizer, anchor_rng, epoch=epoch
    )
    valid_rmse = evaluate_real_ps(model, valid_datasets)
    history.append({
        "epoch": epoch,
        "synthetic_train_rmse": train_rmse,
        "real_ps_valid_rmse": valid_rmse,
    })
    if valid_rmse < best_valid_rmse:
        best_valid_rmse = valid_rmse
        best_state = copy.deepcopy(model.state_dict())
    print(
        f"Epoch {epoch:02d} | synthetic train RMSE: {train_rmse:.4f} | "
        f"real-PS valid RMSE: {valid_rmse:.4f}"
    )

if best_state is not None:
    model.load_state_dict(best_state)
test_rmse = evaluate_real_ps(model, test_datasets)
print(f"Stateful reserved test RMSE: {test_rmse:.4f}")
pd.DataFrame(history)

Epoch 01 | synthetic train RMSE: 123.9054 | real-PS valid RMSE: 349.2477


Epoch 02 | synthetic train RMSE: 113.4192 | real-PS valid RMSE: 323.9241


Epoch 03 | synthetic train RMSE: 105.8811 | real-PS valid RMSE: 251.7903


Epoch 04 | synthetic train RMSE: 105.1481 | real-PS valid RMSE: 210.1467


Epoch 05 | synthetic train RMSE: 101.6256 | real-PS valid RMSE: 229.6526


Stateful reserved test RMSE: 189.6274


,epoch,synthetic_train_rmse,real_ps_valid_rmse
0,1,123.905363,349.247724
1,2,113.419239,323.924094
2,3,105.881093,251.790262
3,4,105.148109,210.146709
4,5,101.625644,229.652579
